# YgoMaster-PackBuilder — Domain Validation Tests

Test offline di `validate_project_definition()`.

Il notebook costruisce un `ProjectDefinition` completamente valido e poi genera copie mutate per verificare tutti gli error code del validator.

Regole coperte:

- unicità di player, pack e opening policy;
- pack rules (`expected_card_count`, `booster_size`, composizione rarità);
- unresolved rarity / Master Duel ID;
- duplicati nello stesso pack;
- riferimenti opening policy;
- main/mini relations;
- mini subset del main;
- stessa rarity tra main e mini;
- sibling mini disgiunti;
- esattamente uno shared staples;
- shared staples senza owner/parent.


## 1. Repository root

In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Impossibile individuare la root della repository.")

print("Repository root:", repo_root)

Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.domain import (
    Card,
    OpeningPolicy,
    PackDefinition,
    PackType,
    PlayerDefinition,
    ProjectDefinition,
    Rarity,
    RarityDistribution,
    SourcePackRef,
    SourceProvider,
    TenPackGuarantee,
    UltraRarePity,
)

from ygomaster_packbuilder.validation import (
    validate_project_definition,
)

print("Imports OK")

Imports OK


## 3. Helper per carte, policy e pack

In [3]:
def card(
    ydk_id,
    rarity,
    *,
    master_duel_id=None,
    name=None,
):
    return Card(
        name=name or f"Card {ydk_id}",
        ydk_id=ydk_id,
        source_rarity=rarity.value,
        master_duel_id=(
            master_duel_id
            if master_duel_id is not None
            else ydk_id + 100000
        ),
        rarity=rarity,
    )


def make_policy(key="configuration-a"):
    return OpeningPolicy(
        key=key,
        normal_slot_distribution=RarityDistribution(
            normal=55.0,
            rare=35.0,
            super_rare=7.5,
            ultra_rare=2.5,
        ),
        ten_pack_guarantee=TenPackGuarantee(
            enabled=True,
            final_slot_distribution=RarityDistribution(
                normal=0.0,
                rare=0.0,
                super_rare=80.0,
                ultra_rare=20.0,
            ),
        ),
        ultra_rare_pity=UltraRarePity(
            enabled=True,
            guaranteed_slot_distribution=RarityDistribution(
                normal=0.0,
                rare=0.0,
                super_rare=0.0,
                ultra_rare=100.0,
            ),
        ),
    )


def source_ref(external_id):
    return SourcePackRef(
        provider=SourceProvider.YGOPRODECK,
        external_id=external_id,
    )


print("Helpers ready")

Helpers ready


## 4. Main pack valido da 100 carte

In [4]:
main_cards = []

for ydk_id in range(1, 51):
    main_cards.append(card(ydk_id, Rarity.N))

for ydk_id in range(51, 76):
    main_cards.append(card(ydk_id, Rarity.R))

for ydk_id in range(76, 91):
    main_cards.append(card(ydk_id, Rarity.SR))

for ydk_id in range(91, 101):
    main_cards.append(card(ydk_id, Rarity.UR))

assert len(main_cards) == 100

main_pack = PackDefinition(
    key="player-main",
    name="Player Main",
    description="Valid main pack",
    type=PackType.PERSONAL_MAIN,
    source=source_ref("main"),
    expected_card_count=100,
    booster_size=7,
    opening_policy_key="configuration-a",
    owner_key="player",
    parent_pack_key=None,
    cards=main_cards,
)

print("Main pack ready")

Main pack ready


## 5. Tre mini-pack validi, subset e disgiunti

In [5]:
def clone_from_main(ids):
    lookup = {
        c.ydk_id: c
        for c in main_pack.cards
    }

    return [
        deepcopy(lookup[ydk_id])
        for ydk_id in ids
    ]


mini_1_ids = (
    list(range(1, 11))
    + list(range(51, 56))
    + list(range(76, 79))
    + list(range(91, 93))
)

mini_2_ids = (
    list(range(11, 21))
    + list(range(56, 61))
    + list(range(79, 82))
    + list(range(93, 95))
)

mini_3_ids = (
    list(range(21, 31))
    + list(range(61, 66))
    + list(range(82, 85))
    + list(range(95, 97))
)


def make_mini(key, external_id, ids):
    return PackDefinition(
        key=key,
        name=key,
        description="Valid mini pack",
        type=PackType.PERSONAL_MINI,
        source=source_ref(external_id),
        expected_card_count=20,
        booster_size=3,
        opening_policy_key="configuration-a",
        owner_key="player",
        parent_pack_key="player-main",
        cards=clone_from_main(ids),
    )


mini_1 = make_mini("player-mini-1", "mini-1", mini_1_ids)
mini_2 = make_mini("player-mini-2", "mini-2", mini_2_ids)
mini_3 = make_mini("player-mini-3", "mini-3", mini_3_ids)

assert len(mini_1.cards) == 20
assert len(mini_2.cards) == 20
assert len(mini_3.cards) == 20

print("Mini packs ready")

Mini packs ready


## 6. Shared staples valido da 200 carte

In [6]:
shared_cards = []

next_id = 10001

for _ in range(100):
    shared_cards.append(card(next_id, Rarity.N))
    next_id += 1

for _ in range(60):
    shared_cards.append(card(next_id, Rarity.R))
    next_id += 1

for _ in range(20):
    shared_cards.append(card(next_id, Rarity.SR))
    next_id += 1

for _ in range(20):
    shared_cards.append(card(next_id, Rarity.UR))
    next_id += 1

assert len(shared_cards) == 200

shared_pack = PackDefinition(
    key="shared-staples",
    name="Shared Staples",
    description="Valid shared pack",
    type=PackType.SHARED_STAPLES,
    source=source_ref("shared"),
    expected_card_count=200,
    booster_size=11,
    opening_policy_key="configuration-a",
    owner_key=None,
    parent_pack_key=None,
    cards=shared_cards,
)

print("Shared pack ready")

Shared pack ready


## 7. ProjectDefinition valido

In [7]:
player = PlayerDefinition(
    key="player",
    display_name="Player",
    main_pack_key="player-main",
    mini_pack_keys=[
        "player-mini-1",
        "player-mini-2",
        "player-mini-3",
    ],
)

valid_project = ProjectDefinition(
    name="Valid Project",
    players=[player],
    packs=[
        main_pack,
        mini_1,
        mini_2,
        mini_3,
        shared_pack,
    ],
    opening_policies=[
        make_policy("configuration-a"),
    ],
)

report = validate_project_definition(valid_project)

assert report.is_valid
assert report.errors == []
assert report.issues == []

print("Valid project accepted")

Valid project accepted


## 8. Helper per verificare un error code

In [8]:
def assert_error_code(
    project,
    expected_code,
):
    report = validate_project_definition(project)

    codes = [
        issue.code
        for issue in report.errors
    ]

    assert expected_code in codes, (
        f"Expected error code {expected_code!r}, "
        f"got {codes}"
    )

    print(expected_code, "OK")


print("Assertion helper ready")

Assertion helper ready


## 9. duplicate-player-key

In [9]:
project = deepcopy(valid_project)
project.players.append(
    deepcopy(project.players[0])
)

assert_error_code(
    project,
    "duplicate-player-key",
)

duplicate-player-key OK


## 10. duplicate-pack-key

In [10]:
project = deepcopy(valid_project)
duplicate = deepcopy(project.packs[0])
project.packs.append(duplicate)

assert_error_code(
    project,
    "duplicate-pack-key",
)

duplicate-pack-key OK


## 11. duplicate-opening-policy-key

In [11]:
project = deepcopy(valid_project)
project.opening_policies.append(
    deepcopy(project.opening_policies[0])
)

assert_error_code(
    project,
    "duplicate-opening-policy-key",
)

duplicate-opening-policy-key OK


## 12. invalid-expected-card-count

In [12]:
project = deepcopy(valid_project)
project.packs[0].expected_card_count = 999

assert_error_code(
    project,
    "invalid-expected-card-count",
)

invalid-expected-card-count OK


## 13. invalid-booster-size

In [13]:
project = deepcopy(valid_project)
project.packs[0].booster_size = 999

assert_error_code(
    project,
    "invalid-booster-size",
)

invalid-booster-size OK


## 14. invalid-card-count

In [14]:
project = deepcopy(valid_project)
project.packs[0].cards.pop()

assert_error_code(
    project,
    "invalid-card-count",
)

invalid-card-count OK


## 15. unknown-opening-policy

In [15]:
project = deepcopy(valid_project)
project.packs[0].opening_policy_key = "missing-policy"

assert_error_code(
    project,
    "unknown-opening-policy",
)

unknown-opening-policy OK


## 16. duplicate-card

In [16]:
project = deepcopy(valid_project)

project.packs[0].cards[1].ydk_id = (
    project.packs[0].cards[0].ydk_id
)

assert_error_code(
    project,
    "duplicate-card",
)

duplicate-card OK


## 17. unresolved-rarity

In [17]:
project = deepcopy(valid_project)
project.packs[0].cards[0].rarity = None

assert_error_code(
    project,
    "unresolved-rarity",
)

unresolved-rarity OK


## 18. unresolved-master-duel-id

In [18]:
project = deepcopy(valid_project)
project.packs[0].cards[0].master_duel_id = None

assert_error_code(
    project,
    "unresolved-master-duel-id",
)

unresolved-master-duel-id OK


## 19. invalid-rarity-composition

In [19]:
project = deepcopy(valid_project)

project.packs[0].cards[0].rarity = Rarity.UR

assert_error_code(
    project,
    "invalid-rarity-composition",
)

invalid-rarity-composition OK


## 20. invalid-mini-pack-count

In [20]:
project = deepcopy(valid_project)

project.players[0].mini_pack_keys.pop()

assert_error_code(
    project,
    "invalid-mini-pack-count",
)

invalid-mini-pack-count OK


## 21. duplicate-mini-pack-reference

In [21]:
project = deepcopy(valid_project)

project.players[0].mini_pack_keys = [
    "player-mini-1",
    "player-mini-1",
    "player-mini-3",
]

assert_error_code(
    project,
    "duplicate-mini-pack-reference",
)

duplicate-mini-pack-reference OK


## 22. missing-main-pack

In [22]:
project = deepcopy(valid_project)

project.players[0].main_pack_key = "missing-main"

report = validate_project_definition(project)

codes = [issue.code for issue in report.errors]

assert "missing-main-pack" in codes

# Il validator termina la validazione relazionale
# di questo player dopo il main mancante.
print("missing-main-pack OK")

missing-main-pack OK


## 23. invalid-main-pack-type

In [23]:
project = deepcopy(valid_project)

project.packs[0].type = PackType.PERSONAL_MINI

assert_error_code(
    project,
    "invalid-main-pack-type",
)

invalid-main-pack-type OK


## 24. invalid-main-pack-owner

In [24]:
project = deepcopy(valid_project)

project.packs[0].owner_key = "someone-else"

assert_error_code(
    project,
    "invalid-main-pack-owner",
)

invalid-main-pack-owner OK


## 25. main-pack-has-parent

In [25]:
project = deepcopy(valid_project)

project.packs[0].parent_pack_key = "unexpected-parent"

assert_error_code(
    project,
    "main-pack-has-parent",
)

main-pack-has-parent OK


## 26. missing-mini-pack

In [26]:
project = deepcopy(valid_project)

project.players[0].mini_pack_keys[0] = "missing-mini"

assert_error_code(
    project,
    "missing-mini-pack",
)

missing-mini-pack OK


## 27. invalid-mini-pack-type

In [27]:
project = deepcopy(valid_project)

mini = next(
    pack
    for pack in project.packs
    if pack.key == "player-mini-1"
)

mini.type = PackType.PERSONAL_MAIN

assert_error_code(
    project,
    "invalid-mini-pack-type",
)

invalid-mini-pack-type OK


## 28. invalid-mini-pack-owner

In [28]:
project = deepcopy(valid_project)

mini = next(
    pack
    for pack in project.packs
    if pack.key == "player-mini-1"
)

mini.owner_key = "someone-else"

assert_error_code(
    project,
    "invalid-mini-pack-owner",
)

invalid-mini-pack-owner OK


## 29. invalid-mini-pack-parent

In [29]:
project = deepcopy(valid_project)

mini = next(
    pack
    for pack in project.packs
    if pack.key == "player-mini-1"
)

mini.parent_pack_key = "wrong-parent"

assert_error_code(
    project,
    "invalid-mini-pack-parent",
)

invalid-mini-pack-parent OK


## 30. mini-card-not-in-main

In [30]:
project = deepcopy(valid_project)

mini = next(
    pack
    for pack in project.packs
    if pack.key == "player-mini-1"
)

mini.cards[0].ydk_id = 999999

assert_error_code(
    project,
    "mini-card-not-in-main",
)

mini-card-not-in-main OK


## 31. mini-card-rarity-mismatch

In [31]:
project = deepcopy(valid_project)

mini = next(
    pack
    for pack in project.packs
    if pack.key == "player-mini-1"
)

mini.cards[0].rarity = Rarity.UR

assert_error_code(
    project,
    "mini-card-rarity-mismatch",
)

mini-card-rarity-mismatch OK


## 32. mini-pack-overlap

In [32]:
project = deepcopy(valid_project)

mini_1 = next(
    pack
    for pack in project.packs
    if pack.key == "player-mini-1"
)

mini_2 = next(
    pack
    for pack in project.packs
    if pack.key == "player-mini-2"
)

# Manteniamo la rarity corretta e introduciamo
# un overlap reale tra sibling mini.
mini_2.cards[0] = deepcopy(mini_1.cards[0])

assert_error_code(
    project,
    "mini-pack-overlap",
)

mini-pack-overlap OK


## 33. invalid-shared-staples-count: zero

In [33]:
project = deepcopy(valid_project)

project.packs = [
    pack
    for pack in project.packs
    if pack.type is not PackType.SHARED_STAPLES
]

assert_error_code(
    project,
    "invalid-shared-staples-count",
)

invalid-shared-staples-count OK


## 34. invalid-shared-staples-count: due

In [34]:
project = deepcopy(valid_project)

extra_shared = deepcopy(
    next(
        pack
        for pack in project.packs
        if pack.type is PackType.SHARED_STAPLES
    )
)

extra_shared.key = "shared-staples-2"

project.packs.append(extra_shared)

assert_error_code(
    project,
    "invalid-shared-staples-count",
)

invalid-shared-staples-count OK


## 35. shared-pack-has-owner

In [35]:
project = deepcopy(valid_project)

shared = next(
    pack
    for pack in project.packs
    if pack.type is PackType.SHARED_STAPLES
)

shared.owner_key = "player"

assert_error_code(
    project,
    "shared-pack-has-owner",
)

shared-pack-has-owner OK


## 36. shared-pack-has-parent

In [36]:
project = deepcopy(valid_project)

shared = next(
    pack
    for pack in project.packs
    if pack.type is PackType.SHARED_STAPLES
)

shared.parent_pack_key = "player-main"

assert_error_code(
    project,
    "shared-pack-has-parent",
)

shared-pack-has-parent OK


## 37. Più errori contemporaneamente

Il validator deve raccogliere più problemi nello stesso report.


In [37]:
project = deepcopy(valid_project)

project.packs[0].booster_size = 999
project.packs[0].cards[0].master_duel_id = None

shared = next(
    pack
    for pack in project.packs
    if pack.type is PackType.SHARED_STAPLES
)

shared.owner_key = "player"

report = validate_project_definition(project)

codes = {
    issue.code
    for issue in report.errors
}

assert {
    "invalid-booster-size",
    "unresolved-master-duel-id",
    "shared-pack-has-owner",
}.issubset(codes)

assert not report.is_valid

print("Multiple-error accumulation OK")
print(sorted(codes))

Multiple-error accumulation OK
['invalid-booster-size', 'shared-pack-has-owner', 'unresolved-master-duel-id']
